In [1]:
from dist_s1_enumerator import get_mgrs_table

In [2]:
df_mgrs = get_mgrs_table()
df_mgrs.head()

,mgrs_tile_id,utm_epsg,utm_wkt,geometry
0,01FBE,32701,"MULTIPOLYGON(((199980 4500040,199980 4390240,3...","MULTIPOLYGON (((-179.63379 -49.62222, -179.688..."
1,01FBF,32701,"MULTIPOLYGON(((199980 4600000,199980 4490200,3...","MULTIPOLYGON (((-179.58653 -48.72398, -179.638..."
2,01GBH,32701,"MULTIPOLYGON(((199980 4800040,199980 4690240,3...","MULTIPOLYGON (((-179.49865 -46.9259, -179.5458..."
3,01GDM,32701,"MULTIPOLYGON(((399960 5200000,399960 5090200,5...","POLYGON ((-178.23431 -43.34619, -178.25485 -44..."
4,01GEL,32701,"MULTIPOLYGON(((499980 5100040,499980 4990240,6...","POLYGON ((-177.00025 -44.25288, -177.00025 -45..."


In [3]:
MGRS_TILE_ID = '20NLJ'


In [4]:
df_one_tile = df_mgrs[df_mgrs['mgrs_tile_id'] == MGRS_TILE_ID].reset_index(drop=True)
bounds = df_one_tile.geometry.total_bounds.tolist()
bounds

[-64.8006173182651, 2.62439196997674, -63.8114234901165, 3.61869306336647]

In [5]:
import ee

# 1. Initialize Earth Engine
try:
    ee.Initialize(project='opera-one')
except Exception:
    ee.Authenticate()
    ee.Initialize()

# -------------------------------------------------------------------------
# 2. CONFIGURATION
# -------------------------------------------------------------------------
# Ensure 'bounds' and 'MGRS_TILE_ID' are defined in your environment
roi = ee.Geometry.Rectangle(bounds)
scale = 10 
drive_folder = 'RADD_MultiBand_Exports'

# -------------------------------------------------------------------------
# 3. SELECT LAYERS
# -------------------------------------------------------------------------
base_col = ee.ImageCollection('projects/radar-wur/raddalert/v1')

# A. Get the latest Alert
latest_alert = ee.Image(base_col.filterBounds(roi)
                         .filterMetadata('layer', 'contains', 'alert')
                         .sort('version_date', False)
                         .first())

# B. Get the Forest Baseline (Robust Mosaic)
baseline_col = base_col.filterMetadata('layer', 'equals', 'forest_baseline').filterBounds(roi)

# We check the size of the collection on the server side
has_baseline = baseline_col.size().gt(0)

# Use ee.Algorithms.If to return a mosaicked image OR a constant 0 image
# This ensures we ALWAYS have a band named 'Forest_Baseline'
clean_baseline = ee.Image(ee.Algorithms.If(
    has_baseline,
    baseline_col.mosaic().rename('Forest_Baseline').toInt16(),
    ee.Image.constant(0).rename('Forest_Baseline').toInt16().clip(roi)
))

# -------------------------------------------------------------------------
# 4. STACK BANDS
# -------------------------------------------------------------------------
# Final safety check: if latest_alert is null, the script should stop
if latest_alert.getInfo() is not None:
    # Stack Alert, Date, and our guaranteed Baseline
    export_img = latest_alert.select(['Alert', 'Date']).addBands(clean_baseline)
    
    # -------------------------------------------------------------------------
    # 5. EXECUTE EXPORT
    # -------------------------------------------------------------------------
    version_date = latest_alert.get('version_date').getInfo() or 'latest'
    task_description = f'RADD_FullStack_{version_date}_{MGRS_TILE_ID}'.replace('.', '_')

    print(f"Submitting Export for Tile: {MGRS_TILE_ID}")
    
    task = ee.batch.Export.image.toDrive(
        image=export_img,
        description=task_description,
        folder=drive_folder,
        region=roi,
        scale=scale,
        maxPixels=1e13,
        crs='EPSG:4326',
        fileFormat='GeoTIFF',
        skipEmptyTiles=True
    )

    task.start()
    print(f"Task started. Check progress: https://code.earthengine.google.com/tasks")
else:
    print("Execution halted: No RADD alert found for these bounds.")

Submitting Export for Tile: 20NLJ
Task started. Check progress: https://code.earthengine.google.com/tasks


In [6]:
task.status()

{'state': 'READY',
 'description': 'RADD_FullStack_2026-02-15_20NLJ',
 'priority': 100,
 'creation_timestamp_ms': 1771547753196,
 'update_timestamp_ms': 1771547753196,
 'start_timestamp_ms': 0,
 'task_type': 'EXPORT_IMAGE',
 'id': 'Z4DCWPO2UMH3CMVWPQR2HTEH',
 'name': 'projects/opera-one/operations/Z4DCWPO2UMH3CMVWPQR2HTEH'}